In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/14 15:56:46 WARN Utils: Your hostname, DESKTOP-OIMG5OU, resolves to a loopback address: 127.0.1.1; using 192.168.178.151 instead (on interface eth0)
26/07/14 15:56:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/14 15:56:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_green = spark.read.parquet("../data/pq/green/*/*")

26/07/14 15:57:05 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/pq/green/*/*.
java.io.FileNotFoundException: File ../data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Resol

In [4]:
df_yellow = spark.read.parquet("../data/pq/yellow/*/*")

26/07/14 15:57:16 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/pq/yellow/*/*.
java.io.FileNotFoundException: File ../data/pq/yellow/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Res

In [ ]:
df_yellow.printSchema()

In [ ]:
df_green.show()

In [ ]:
set(df_yellow.columns) & set(df_green.columns)

In [5]:
df_green = df_green \
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime") \
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")

In [6]:
df_yellow = df_yellow \
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime") \
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")

In [7]:
common_columns = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_columns.append(col)

In [8]:
from pyspark.sql import functions as F

In [9]:
df_green_selected = df_green \
    .select(common_columns) \
    .withColumn("service_type", F.lit("green"))

In [10]:
df_yellow_selected = df_yellow \
    .select(common_columns) \
    .withColumn("service_type", F.lit("yellow"))

In [11]:
df_trips_data = df_green_selected.unionAll(df_yellow_selected)

In [12]:
df_trips_data.groupBy("service_type").count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [13]:
df_trips_data.registerTempTable('trips_data')

/home/penny_dev/projects/de-zoomcamp/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [16]:
df_result = spark.sql("""
SELECT
    -- Grouping dimensions
    PULocationID as revenue_zone,
    date_trunc('month', pickup_datetime) as revenue_month,
    service_type,
    -- Revenue calculation
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,

    -- Additional metrics for operational analysis
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance

FROM 
    trips_data
GROUP BY 
    1, 2, 3
""")

In [ ]:
df_result.show()

In [ ]:
df_result.coalesce(1).write.parquet("../data/reports/revenue", mode="overwrite")

26/07/14 18:02:24 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 477710 ms exceeds timeout 120000 ms
26/07/14 18:02:26 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.rpc.RpcTimeoutException: Future timed out after [10000 milliseconds]. This timeout is controlled by spark.executor.heartbeatInterval
	at org.apache.spark.rpc.RpcTimeout.org$apache$spark$rpc$RpcTimeout$$createRpcTimeoutException(RpcTimeout.scala:47)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:62)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:58)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:76)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1526)
	at org.apache.spark.ex